# E6: code notebook cũ, nâng giới hạn từ 10 lên 30 epoch
Dùng trainer/loop cũ, không Tee, không custom Trainer. Giữ hyperparameter và scheduler cũ.
Tự chọn recovery đã xác minh có tiến độ cao hơn giữa folder gốc và folder mở rộng trước đó.
Mục tiêu TỔNG 30 epoch: đã xong 10 thì chạy tiếp 20 epoch còn lại; không chia mỗi lần 10 epoch. Chỉ dừng sớm khi tới deadline phiên.
E6 giữ local recovery mỗi 1800 batches; HF tại epoch end, mốc một giờ giữa epoch, hoặc deadline.
Bật GPU, Internet; HF_TOKEN ghi 13ffa/deep, HF_TOKEN2 đọc iraprp/deeplearning khi repo mới chưa có recovery.
Gắn ROD Final. E3 cần Input bundle nếu cả hai repo chưa có recovery.
HF lưu tiếp vào `13ffa/deep/Faster R-CNN/`. Repo cũ chỉ đọc. Không chạy đồng thời notebook khác cho cùng model.


In [ ]:
%pip -q install 'huggingface_hub>=0.34,<2' 'pyyaml>=6.0' 'torchmetrics==1.6.3' 'pycocotools>=2.0.7'

In [ ]:
HF_FOLDER = 'Faster R-CNN'
print('HF checkpoint folder:', HF_FOLDER, flush=True)

In [ ]:
from pathlib import Path
import yaml
DATA_ROOT = Path('/kaggle/input/datasets/nguyenkin/deeplearningg/rod_detection_final')
if not DATA_ROOT.is_dir():
    candidates = [p for p in Path('/kaggle/input').rglob('data.yaml') if 'rod_detection_final' in str(p)]
    if len(candidates) != 1:
        raise FileNotFoundError('Không thấy ROD Final. Gắn dataset deeplearningg vào notebook. Candidate: ' + str(candidates[:5]))
    DATA_ROOT = candidates[0].parent
source_yaml = DATA_ROOT / 'data.yaml'
if not source_yaml.exists():
    raise FileNotFoundError('Thiếu data.yaml ở ' + str(DATA_ROOT))
source_cfg = yaml.safe_load(source_yaml.read_text(encoding='utf-8'))
class_names = source_cfg['names']
if isinstance(class_names, dict):
    class_names = [str(class_names[i] if i in class_names else class_names[str(i)]) for i in range(len(class_names))]
else:
    class_names = [str(x) for x in class_names]
if len(class_names) != 25:
    raise RuntimeError('Phải đúng 25 classes, hiện là: ' + str(len(class_names)))

def find_images(split, candidates):
    candidates = [str(source_cfg.get(split, ''))] + candidates
    for relative in candidates:
        if not relative:
            continue
        d = Path(relative)
        if not d.is_absolute():
            d = (DATA_ROOT / d).resolve()
        if d.is_dir() and any(d.glob('*.jpg')) or d.is_dir() and any(d.glob('*.png')) or d.is_dir() and any(d.glob('*.jpeg')):
            return d
    raise FileNotFoundError(f'Không tìm được images cho split {split} tại {DATA_ROOT}')
TRAIN_IMAGES = find_images('train', ['train/images', 'images/train'])
VAL_IMAGES = find_images('val', ['valid/images', 'val/images', 'images/val', 'images/valid'])
TEST_IMAGES = None  # TEST giữ kín, KHÔNG truy cập khi train.
working_yaml = Path('/kaggle/working/rod_trainval.yaml')
working_yaml.write_text(yaml.safe_dump({
    'path': str(DATA_ROOT),
    'train': str(TRAIN_IMAGES),
    'val': str(VAL_IMAGES),
    'nc': 25,
    'names': class_names,
}, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('Dataset:', DATA_ROOT, '\nTRAIN:', TRAIN_IMAGES, '\nVAL:', VAL_IMAGES, '\nClasses:', len(class_names))

EXPECTED_ROD_NAMES = ['Bike', 'Building', 'Car', 'Person', 'Stairs', 'Traffic sign', 'Electrical Pole', 'Road', 'Motorcycle', 'Dustbin', 'Dog', 'Manhole', 'Tree', 'Guard rail', 'Pedestrian crosswalk', 'Truck', 'Bus', 'Bench', 'Traffic Cone', 'Fire hydrant', 'Teraffic Barrel', 'Plant Pot', 'Electrical Box', 'Chair', 'Bicycle Rack']

if class_names != EXPECTED_ROD_NAMES or source_cfg.get('nc') != 25:
    raise RuntimeError('ROD Final class ID/order không khớp bản freeze')
for split_name, images_dir, expected_count in [('TRAIN', TRAIN_IMAGES, 18151), ('VALID', VAL_IMAGES, 3318)]:
    image_files = [p for p in images_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')]
    labels_dir = images_dir.parent / 'labels'
    if len(image_files) != expected_count or not labels_dir.is_dir():
        raise RuntimeError(f'{split_name} image/label directory count không khớp ROD Final A4')
    image_stems = {p.stem for p in image_files}
    label_stems = {p.stem for p in labels_dir.glob('*.txt')}
    if image_stems != label_stems:
        raise RuntimeError(f'{split_name} image/label không khớp từng file')
print('ROD Final A4 TRAIN/VALID image-label pairs và 25 class: PASS')


In [ ]:
"""Embedded in both Kaggle notebooks. No local import is required at runtime."""

import hashlib
import json
import os
import re
import shutil
import time
import uuid
from pathlib import Path


HF_REPO_ID = "13ffa/deep"
HF_SECRET_NAME = "HF_TOKEN"


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for part in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True), encoding="utf-8")
    os.replace(temp, path)


def validate_progress(manifest, folder, model, config_hash, dataset_hash, max_epoch=30):
    if manifest.get("schema") != 2 or manifest.get("folder") != folder or manifest.get("model") != model:
        raise RuntimeError("HF manifest thuộc mô hình/folder khác hoặc sai schema")
    if manifest.get("config_sha256") != config_hash or manifest.get("dataset_sha256") != dataset_hash:
        raise RuntimeError("HF checkpoint khác config/dataset; dừng, không train lại")
    epoch = manifest.get("completed_epoch")
    if type(epoch) is not int or not 0 <= epoch <= max_epoch:
        raise RuntimeError("HF completed_epoch không hợp lệ")
    legacy_ten = (max_epoch == 30 and epoch == 10 and manifest.get("is_complete") is True
                  and manifest.get("target_epochs", 10) == 10)
    if not legacy_ten and manifest.get("is_complete") is not (epoch == max_epoch):
        raise RuntimeError("HF trạng thái COMPLETE không khớp epoch")
    if not isinstance(manifest.get("sha256"), dict) or not manifest["sha256"]:
        raise RuntimeError("HF manifest thiếu checksum")
    return epoch


class HFStore:
    """Immutable snapshot first, latest pointer second. Failed upload cannot advance latest."""

    def __init__(self, api, token, repo_type, folder, model, config_hash, dataset_hash, work, max_epoch=30, repo_id=None, compact=False):
        if folder not in ("YOLO 11n", "YOLO 11s", "Faster R-CNN", "E3_30epochs", "E4_30epochs", "E6_30epochs"):
            raise ValueError("Unknown experiment folder")
        if type(max_epoch) is not int or max_epoch not in (10, 30):
            raise ValueError("Invalid epoch budget")
        self.max_epoch = max_epoch
        self.compact = compact
        self.repo_id = HF_REPO_ID if repo_id is None else repo_id
        self.api, self.token, self.repo_type = api, token, repo_type
        self.folder, self.model = folder, model
        self.config_hash, self.dataset_hash = config_hash, dataset_hash
        self.work = Path(work)
        self.work.mkdir(parents=True, exist_ok=True)

    def _files(self):
        return set(self.api.list_repo_files(repo_id=self.repo_id, repo_type=self.repo_type, token=self.token))

    def _download(self, remote):
        from huggingface_hub import hf_hub_download
        return Path(hf_hub_download(repo_id=self.repo_id, repo_type=self.repo_type,
                                    filename=remote, token=self.token, local_dir=str(self.work / "download"),
                                    force_download=remote.endswith("latest.json")))

    def restore(self):
        files = self._files()  # connection/auth errors must propagate
        prefix = self.folder + "/"
        relevant = [name for name in files if name.startswith(prefix)]
        pointer_name = prefix + "latest.json"
        if pointer_name not in files:
            if relevant:
                raise RuntimeError("HF folder có dữ liệu nhưng thiếu latest.json; dừng để kiểm tra thủ công")
            return None
        pointer = json.loads(self._download(pointer_name).read_text(encoding="utf-8"))
        snap = pointer.get("snapshot")
        if not isinstance(snap, str) or not snap.startswith(prefix + "snapshots/") or ".." in snap:
            raise RuntimeError("HF latest pointer không hợp lệ")
        manifest_name = snap + "/manifest.json"
        if manifest_name not in files:
            raise RuntimeError("HF latest pointer trỏ tới snapshot thiếu manifest")
        manifest_path = self._download(manifest_name)
        if sha256_file(manifest_path) != pointer.get("manifest_sha256"):
            raise RuntimeError("HF manifest SHA256 không khớp latest pointer")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        validate_progress(manifest, self.folder, self.model, self.config_hash, self.dataset_hash, self.max_epoch)
        if manifest.get("snapshot") != snap:
            raise RuntimeError("HF snapshot ID không nhất quán")
        local = {}
        for name, expected in manifest["sha256"].items():
            if name != Path(name).name or snap + "/" + name not in files:
                raise RuntimeError("HF snapshot thiếu artifact hoặc tên file không hợp lệ")
            path = self._download(snap + "/" + name)
            if sha256_file(path) != expected:
                raise RuntimeError("HF artifact SHA256 không khớp: " + name)
            local[name] = path
        for alias, target in manifest.get("artifact_aliases", {}).items():
            if (alias not in ("best.pt", "best_weights.pth") or target not in ("checkpoint.pt", "recovery_full.pt")
                or target not in local or alias in local):
                raise RuntimeError("HF artifact alias không hợp lệ")
            local[alias] = local[target]
        return manifest, local

    def _compact_files(self, files):
        """Keep the full resume payload; remove only redundant/unneeded uploads."""
        recovery = "checkpoint.pt" if self.model == "fasterrcnn_resnet50_fpn_v2" else "recovery_full.pt"
        required = ({"checkpoint.pt", "metrics.csv", "config.json", "environment.json"}
                    if recovery == "checkpoint.pt" else
                    {"recovery_full.pt", "results.csv", "args.yaml", "config.json", "environment.json"})
        best = "best_weights.pth" if recovery == "checkpoint.pt" else "best.pt"
        if not required.issubset(files):
            raise RuntimeError("Compact upload thiếu file resume/config/metrics bắt buộc")
        selected = {name: source for name, source in files.items() if name in required | {best}}
        aliases = {}
        if best in selected and sha256_file(selected[best]) == sha256_file(selected[recovery]):
            aliases[best] = recovery
            del selected[best]
        return selected, aliases

    def _prune_compact(self, current):
        """Bound the live tree, never squash shared repo history during training.

        Only compact-v1 snapshots of this exact folder/config/dataset are owned.
        Unknown snapshots, source repositories, other models, and future progress
        are never deleted. A conditional commit fails if another writer commits.
        """
        if self.repo_id != HF_REPO_ID:
            raise RuntimeError("Refusing retention writes to source repository")
        head = self.api.repo_info(repo_id=self.repo_id, repo_type=self.repo_type, token=self.token).sha
        files = set(self.api.list_repo_files(repo_id=self.repo_id, repo_type=self.repo_type,
                                             token=self.token, revision=head))
        pointer_name = self.folder + "/latest.json"
        pointer = json.loads(self._download(pointer_name).read_text(encoding="utf-8"))
        if (pointer.get("snapshot") != current["snapshot"] or
            pointer.get("manifest_sha256") != sha256_file(self._download(current["snapshot"] + "/manifest.json"))):
            raise RuntimeError("Latest changed; retention skipped")
        prefix = self.folder + "/snapshots/"
        records = {}
        for name in sorted(files):
            if not name.startswith(prefix) or not name.endswith("/manifest.json"):
                continue
            snap = name.rsplit("/", 1)[0]
            if snap.count("/") != 2:
                continue
            m = json.loads(self._download(name).read_text(encoding="utf-8"))
            if (m.get("storage_policy") != "compact-v1" or m.get("folder") != self.folder or
                m.get("snapshot") != snap or m.get("model") != self.model or
                m.get("config_sha256") != self.config_hash or m.get("dataset_sha256") != self.dataset_hash):
                continue
            validate_progress(m, self.folder, self.model, self.config_hash, self.dataset_hash, self.max_epoch)
            if type(m.get("next_sample", 0)) is not int or m.get("next_sample", 0) < 0:
                raise RuntimeError("Invalid progress during retention")
            actual = {p for p in files if p.startswith(snap + "/")}
            expected = {snap + "/manifest.json"}
            for artifact in m["sha256"]:
                if artifact != Path(artifact).name or "/" in artifact or "\\" in artifact:
                    raise RuntimeError("Unsafe artifact path during retention")
                expected.add(snap + "/" + artifact)
            if actual == expected:
                records[snap] = m
        if current["snapshot"] not in records:
            raise RuntimeError("Current snapshot incomplete; no retention deletion")
        def progress(s):
            return records[s]["completed_epoch"], records[s].get("next_sample", 0)
        keep = {current["snapshot"]}
        for epoch in (10, 20, 30):
            complete = [s for s, m in records.items() if m["completed_epoch"] == epoch and m.get("next_sample", 0) == 0]
            if complete:
                keep.add(max(complete, key=lambda s: (s == current["snapshot"], s)))
        # Defensive fallback if an early/partial current snapshot has no best yet.
        best = [s for s, m in records.items() if
                {"best.pt", "best_weights.pth"} & (set(m["sha256"]) | set(m.get("artifact_aliases", {})))]
        if best and not ({"best.pt", "best_weights.pth"} &
                         (set(current["sha256"]) | set(current.get("artifact_aliases", {})))):
            keep.add(max(best, key=lambda s: (progress(s), s)))
        # Never prune a snapshot at more advanced progress, even if not latest yet.
        obsolete = {s for s in records if s not in keep and progress(s) <= progress(current["snapshot"])}
        paths = sorted(p for p in files if p.rsplit("/", 1)[0] in obsolete)
        if not paths:
            return
        from huggingface_hub import CommitOperationDelete
        self.api.create_commit(repo_id=self.repo_id, repo_type=self.repo_type, token=self.token,
            parent_commit=head, revision="main", operations=[CommitOperationDelete(path_in_repo=p) for p in paths],
            commit_message=f"{self.folder}: prune verified compact snapshots; keep latest/best/milestones")
        print("HF retention:", self.folder, "removed snapshots=", len(obsolete),
              "; retained latest/best/milestones. Git history unchanged.", flush=True)

    def _retry(self, action):
        for attempt in range(3):
            try:
                return action()
            except Exception as exc:
                response = getattr(exc, "response", None)
                status = getattr(response, "status_code", None)
                detail = getattr(exc, "server_message", None)
                if not detail and response is not None:
                    try:
                        body = response.json()
                        if isinstance(body, dict):
                            detail = body.get("error") or body.get("message")
                    except (ValueError, AttributeError):
                        pass
                # Some HF/Xet errors have only a plain-text HTTP response.
                body_text = getattr(response, "text", "") if response is not None else ""
                detail = str(detail or exc)
                if body_text and str(body_text) not in detail:
                    detail += "\nResponse: " + str(body_text)
                if self.token:
                    detail = detail.replace(self.token, "[REDACTED]")
                detail = re.sub(r"hf_[A-Za-z0-9_]+", "[REDACTED]", detail)
                detail = re.sub(r"Bearer\s+\S+", "Bearer [REDACTED]", detail, flags=re.I)
                detail = re.sub(r"https?://[^\s]+", "[URL REDACTED]", detail)[:6000]
                diagnostic = f"{type(exc).__name__}; HTTP {status}; {detail}"
                record = {"time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                          "folder": self.folder, "operation": getattr(self, "_upload_operation", "upload"),
                          "attempt": attempt + 1, "error_type": type(exc).__name__,
                          "http_status": status, "server_message": detail,
                          "local_checkpoint_retained": True}
                # Keep diagnostics in Kaggle Output even when Save & Run All fails.
                try:
                    with (self.work / "hf_upload_errors.jsonl").open("a", encoding="utf-8") as log:
                        log.write(json.dumps(record, ensure_ascii=False) + "\n")
                except OSError:
                    print("Không ghi được log HF ra đĩa; xem lỗi bên dưới.", flush=True)
                print("HF upload lỗi", attempt + 1, diagnostic, "; bản local còn ở", self.work, flush=True)
                if attempt == 2:
                    raise RuntimeError("HF upload thất bại; giữ Kaggle Output và KHÔNG đánh dấu đã đồng bộ. "
                                       + diagnostic) from None
                time.sleep(5 * (attempt + 1))

    def save(self, files, completed_epoch, extra=None):
        aliases = {}
        if self.compact:
            files, aliases = self._compact_files(files)
        snap = self.folder + "/snapshots/" + uuid.uuid4().hex
        stage = self.work / "snapshots" / snap.rsplit("/", 1)[-1]
        stage.mkdir(parents=True, exist_ok=False)
        hashes = {}
        for name, source in files.items():
            if name != Path(name).name or not Path(source).is_file():
                raise RuntimeError("Artifact không hợp lệ: " + name)
            target = stage / name
            # Kaggle checkpoint and stage live on the same filesystem. Hardlinks
            # avoid duplicating a large optimizer checkpoint during HF upload.
            # Fall back to a copy if the filesystem does not support hardlinks.
            try:
                os.link(Path(source).resolve(), target)
            except OSError:
                shutil.copy2(source, target)
            hashes[name] = sha256_file(target)
        manifest = {**(extra or {}), "schema": 2, "folder": self.folder, "model": self.model,
                    "snapshot": snap, "config_sha256": self.config_hash,
                    "dataset_sha256": self.dataset_hash, "target_epochs": 30,
                    "completed_epoch": int(completed_epoch), "is_complete": int(completed_epoch) == self.max_epoch,
                    "sha256": hashes}
        if self.compact:
            manifest.update(storage_policy="compact-v1", artifact_aliases=aliases)
        validate_progress(manifest, self.folder, self.model, self.config_hash, self.dataset_hash, self.max_epoch)
        atomic_json(stage / "manifest.json", manifest)
        self._upload_operation = "upload_folder"
        print("HF upload_folder bắt đầu:", self.folder, "epoch", completed_epoch,
              "repo=", self.repo_id, "repo_type=", self.repo_type, "remote=", snap,
              "files=", len(files) + 1,
              "upload_MiB=", round(sum(p.stat().st_size for p in stage.iterdir()) / 1024**2, 2),
              "local=", stage, flush=True)
        self._retry(lambda: self.api.upload_folder(folder_path=str(stage), path_in_repo=snap,
                     repo_id=self.repo_id, repo_type=self.repo_type, token=self.token,
                     commit_message=f"{self.folder}: snapshot epoch {completed_epoch}"))
        # Verify server snapshot before moving pointer. A failed pointer commit leaves previous version valid.
        remote_manifest = self._download(snap + "/manifest.json")
        if sha256_file(remote_manifest) != sha256_file(stage / "manifest.json"):
            raise RuntimeError("HF upload manifest verification failed")
        pointer = self.work / "latest.json"
        atomic_json(pointer, {"snapshot": snap, "manifest_sha256": sha256_file(stage / "manifest.json")})
        self._upload_operation = "upload_latest_pointer"
        print("HF snapshot manifest verified; updating latest:", self.folder, flush=True)
        self._retry(lambda: self.api.upload_file(path_or_fileobj=str(pointer), path_in_repo=self.folder + "/latest.json",
                    repo_id=self.repo_id, repo_type=self.repo_type, token=self.token,
                    commit_message=f"{self.folder}: advance epoch {completed_epoch}"))
        remote_pointer = json.loads(self._download(self.folder + "/latest.json").read_text(encoding="utf-8"))
        if remote_pointer != json.loads(pointer.read_text(encoding="utf-8")):
            raise RuntimeError("HF latest pointer chưa được xác nhận sau upload")
        if self.compact:
            remote_files = self._files()
            if any(snap + "/" + name not in remote_files for name in hashes):
                raise RuntimeError("Uploaded snapshot missing an artifact; retention NOT run")
            try:
                self._prune_compact(manifest)
            except Exception as error:
                # New recovery is durable; cleanup conflicts must not lose it or
                # stop a successful epoch. Retry retention on the next save.
                warning = {"folder": self.folder, "snapshot": snap, "retention": "SKIPPED",
                           "error_type": type(error).__name__,
                           "http_status": getattr(getattr(error, "response", None), "status_code", None)}
                atomic_json(self.work / "retention_warning.json", warning)
                print("WARNING: retention skipped; checkpoint uploaded. Extra snapshots kept:",
                      type(error).__name__, flush=True)
        # Only temporary staging files are removed; the source checkpoint in
        # Kaggle Output and the verified immutable HF snapshot remain intact.
        if stage.resolve().parent != (self.work / "snapshots").resolve():
            raise RuntimeError("Local staging path outside expected snapshots directory")
        for staged_file in stage.iterdir():
            if not staged_file.is_file():
                raise RuntimeError("Unexpected directory in local staging snapshot")
            staged_file.unlink()
        stage.rmdir()
        print("HF snapshot verified:", self.folder, "epoch", completed_epoch,
              "next_sample", manifest.get("next_sample", 0), "local staging cleared; free_GB=",
              round(shutil.disk_usage(self.work).free / (1024 ** 3), 2))
        return manifest


def connect_hf(token):
    from huggingface_hub import HfApi
    if not token or not token.strip():
        raise RuntimeError("Thiếu Kaggle Secret HF_TOKEN")
    api = HfApi(token=token)
    matches = []
    for kind in ("dataset", "model"):
        try:
            api.repo_info(repo_id=HF_REPO_ID, repo_type=kind)
            matches.append(kind)
        except Exception:
            pass
    if len(matches) != 1:
        raise RuntimeError("Không xác định duy nhất repo_type của " + HF_REPO_ID + "; kiểm tra token/quyền/repo")
    return api, matches[0]


import zipfile
from pathlib import PurePosixPath
def checked_member(name):
    if '\\' in name or ':' in name or name.startswith('/'):
        raise ValueError('Unsafe bundle member: ' + name)
    path = PurePosixPath(name)
    if '..' in path.parts or not path.parts:
        raise ValueError('Unsafe bundle member: ' + name)
    return name

def atomic_copy(source, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    temporary = dest.with_suffix(dest.suffix + '.tmp')
    shutil.copyfile(source, temporary)
    os.replace(temporary, dest)

def extract_bundle(path, destination, experiment, hash_file):
    """Only extract the flat, hashed payload of a locally audited bundle."""
    destination = Path(destination)
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or len(names) > 12:
            raise RuntimeError('Duplicate/too many bundle members')
        for entry in archive.infolist():
            checked_member(entry.filename)
            if '/' in entry.filename or entry.is_dir() or entry.file_size > 2 * 1024**3:
                raise RuntimeError('Invalid bundle member')
        manifest = json.loads(archive.read('source_manifest.json'))
        if manifest.get('experiment') != experiment or manifest.get('completed_epoch') != 10:
            raise RuntimeError('Wrong source model or epoch')
        hashes = manifest.get('sha256', {})
        if set(names) != set(hashes) | {'source_manifest.json'} or not hashes:
            raise RuntimeError('Unexpected bundle artifacts')
        if archive.testzip() is not None:
            raise RuntimeError('Bundle CRC failed')
        destination.mkdir(parents=True, exist_ok=True)
        for name, expected in hashes.items():
            target = destination / name
            if target.exists():
                if hash_file(target) != expected:
                    raise RuntimeError('Refusing to overwrite different source artifact')
                continue
            temp = target.with_suffix(target.suffix + '.tmp')
            with archive.open(name) as src, temp.open('wb') as dst:
                shutil.copyfileobj(src, dst)
            if hash_file(temp) != expected:
                temp.unlink()
                raise RuntimeError('Bundle SHA256 failed: ' + name)
            os.replace(temp, target)
    return manifest

def discover_source(input_root, destination, experiment, hash_file):
    """Kaggle may expose the uploaded ZIP as an already extracted directory."""
    root = Path(input_root)
    candidates = list(root.rglob(experiment.lower() + '_resume_epoch010.zip'))
    observed = []
    for path in root.rglob('source_manifest.json'):
        try:
            manifest = json.loads(path.read_text(encoding='utf-8'))
        except (ValueError, OSError):
            continue
        observed.append(str(path) + ' [' + str(manifest.get('experiment')) + ']')
        if manifest.get('experiment') == experiment:
            candidates.append(path)
    if len(candidates) != 1:
        mounted = [p.name for p in root.iterdir()] if root.is_dir() else []
        raise RuntimeError(f'{experiment}: expected exactly one ZIP or extracted source bundle, found {len(candidates)}. '
                           f'Candidates: {[str(p) for p in candidates]}; manifests: {observed}; '
                           f'mounted inputs: {mounted}. Attach the audited {experiment.lower()}_resume_epoch010 bundle. '
                           'No training was started; never restart at epoch 1.')
    path = candidates[0]
    print('Using audited source:', path, flush=True)
    if path.suffix.lower() == '.zip':
        return extract_bundle(path, destination, experiment, hash_file)
    manifest = json.loads(path.read_text(encoding='utf-8'))
    if manifest.get('completed_epoch') != 10:
        raise RuntimeError('Wrong extracted source epoch')
    hashes = manifest.get('sha256', {})
    if not isinstance(hashes, dict) or not hashes or len(hashes) > 10:
        raise RuntimeError('Extracted source manifest missing hashes')
    # Validate every source before copying any file, and prohibit path traversal/symlinks.
    for name, expected in hashes.items():
        checked_member(name)
        if '/' in name or (path.parent / name).resolve().parent != path.parent.resolve():
            raise RuntimeError('Unsafe extracted artifact name')
        source = path.parent / name
        if not source.is_file() or hash_file(source) != expected:
            raise RuntimeError('Extracted source missing artifact or SHA256 mismatch: ' + name)
    for name, expected in hashes.items():
        target = Path(destination) / name
        if target.exists() and hash_file(target) != expected:
            raise RuntimeError('Refusing to overwrite different extracted artifact: ' + name)
        if not target.exists():
            atomic_copy(path.parent / name, target)
        if hash_file(target) != expected:
            raise RuntimeError('Copied artifact SHA256 mismatch: ' + name)
    return manifest


def choose_resume(store, previous_folder, previous_hash):
    # Continue from the destination once it has a verified recovery.
    current = store.restore()
    previous = HFStore(store.api, store.token, store.repo_type, previous_folder,
        store.model, previous_hash, store.dataset_hash, store.work / 'previous', max_epoch=30).restore()
    choices = [item for item in (current, previous) if item is not None]
    if not choices:
        from kaggle_secrets import UserSecretsClient
        try:
            source_token = UserSecretsClient().get_secret('HF_TOKEN2')
        except Exception:
            raise RuntimeError('New repo has no recovery. Enable HF_TOKEN2 with READ access to iraprp/deeplearning; HF_TOKEN writes to 13ffa/deep.') from None
        if not source_token or not source_token.strip():
            raise RuntimeError('Missing read-only source credential HF_TOKEN2')
        for folder, config, suffix in ((store.folder, store.config_hash, 'source'),
                                       (previous_folder, previous_hash, 'source_extension')):
            source = HFStore(store.api, source_token, 'dataset', folder, store.model,
                config, store.dataset_hash, store.work / suffix, max_epoch=30,
                repo_id='iraprp/deeplearning')
            recovered = source.restore()  # Source errors propagate; never silently restart.
            if recovered is not None:
                choices.append(recovered)
        print('Read-only source: iraprp/deeplearning; uploads: 13ffa/deep', flush=True)
    if not choices:
        return None
    selected = max(choices, key=lambda item: (item[0]['completed_epoch'], item[0].get('next_sample',0)))
    print('Verified resume:', selected[0]['folder'], 'completed_epoch=', selected[0]['completed_epoch'],
          'next_sample=', selected[0].get('next_sample',0), flush=True)
    print('Training target: 30 total epochs; completed=', selected[0]['completed_epoch'],
          'remaining=', 30 - selected[0]['completed_epoch'],
          '; continue to 30 or session deadline, no 10-epoch stop', flush=True)
    print('Checkpoint destination: 13ffa/deep/' + store.folder, flush=True)
    return selected

def yolo_resume_directory(checkpoint):
    # Keep the original trainer and restore its historical output directory.
    args = torch.load(checkpoint, map_location='cpu', weights_only=False)['train_args']
    path = Path(args.get('save_dir') or str(Path(args['project']) / args['name']))
    if not path.resolve().is_relative_to(Path('/kaggle/working').resolve()):
        raise RuntimeError('Checkpoint output path is outside /kaggle/working')
    path.mkdir(parents=True, exist_ok=True)
    return path

def load_e3_input():
    source = Path('/kaggle/working/e3_input_recovery')
    origin = discover_source('/kaggle/input', source, 'E3', sha256_file)
    if origin['dataset_yaml_sha256'] != sha256_file(source_yaml) or origin['class_names'] != class_names:
        raise RuntimeError('E3 input differs from ROD Final')
    return ({'completed_epoch':10,'next_sample':0}, {name:source/name for name in origin['sha256']})


In [ ]:
"""Standalone E6 notebook training cell (embedded by build_hf_autoresume_notebooks.py)."""

import csv
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torchvision
from PIL import Image
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader, Dataset, Subset
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms.functional import to_tensor


E6_FOLDER = "Faster R-CNN"
E6_MODEL = "fasterrcnn_resnet50_fpn_v2"
E6_WORK = Path("/kaggle/working/rod_e6_faster_rcnn_hf")
E6_WORK.mkdir(parents=True, exist_ok=True)
E6_EPOCHS, E6_BATCH, E6_BACKUP_BATCHES = 30, 2, 1800
E6_DEADLINE = time.monotonic() + 9.5 * 3600  # reserve ~2.5 h for HF upload / Kaggle shutdown

if not torch.cuda.is_available():
    raise RuntimeError("E6 cần Kaggle GPU; chưa có CUDA")
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.benchmark = False


class RODDataset(Dataset):
    def __init__(self, images_dir):
        self.images_dir = Path(images_dir)
        self.images = sorted(p for p in self.images_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"))
        self.labels_dir = self.images_dir.parent / "labels"
        if not self.images or not self.labels_dir.is_dir():
            raise RuntimeError("ROD images/labels thiếu trong " + str(self.images_dir))
        if any(not (self.labels_dir / (p.stem + ".txt")).is_file() for p in self.images):
            raise RuntimeError("ROD có ảnh thiếu label trong " + str(self.images_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        p = self.images[index]
        with Image.open(p) as image:
            image = image.convert("RGB")
            width, height = image.size
            boxes, labels = [], []
            for row in (self.labels_dir / (p.stem + ".txt")).read_text(encoding="utf-8").splitlines():
                if not row.strip():
                    continue
                fields = row.split()
                if len(fields) != 5:
                    raise RuntimeError("YOLO annotation phải có 5 trường: " + str(p))
                values = [float(x) for x in fields]
                c, x, y, bw, bh = values
                left, top, right, bottom = x - bw / 2, y - bh / 2, x + bw / 2, y + bh / 2
                # Decimal YOLO labels can round an edge at 1.0 to 1.00000000005.
                # Accept only this tiny serialization error, then clip to the image.
                edge_tolerance = 1e-6
                if not (math.isfinite(c) and c.is_integer() and 0 <= c < 25 and
                        all(math.isfinite(v) for v in values[1:]) and 0 < bw <= 1 and 0 < bh <= 1 and
                        -edge_tolerance <= left < right <= 1 + edge_tolerance and
                        -edge_tolerance <= top < bottom <= 1 + edge_tolerance):
                    raise RuntimeError("YOLO class/bbox ngoài phạm vi: " + str(p))
                clipped = [max(0., left) * width, max(0., top) * height,
                           min(1., right) * width, min(1., bottom) * height]
                if clipped[2] <= clipped[0] or clipped[3] <= clipped[1]:
                    raise RuntimeError("YOLO bbox rỗng sau khi xử lý sai số biên: " + str(p))
                boxes.append(clipped)
                labels.append(int(c) + 1)  # Torchvision background 0; ROD classes 1..25.
            b = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
            target = {"boxes": b, "labels": torch.tensor(labels, dtype=torch.int64),
                      "image_id": torch.tensor([index], dtype=torch.int64),
                      "area": (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]),
                      "iscrowd": torch.zeros(len(labels), dtype=torch.int64)}
            return to_tensor(image), target


def collate(batch):
    return tuple(zip(*batch))


train_ds, val_ds = RODDataset(TRAIN_IMAGES), RODDataset(VAL_IMAGES)
if len(train_ds) != 18151 or len(val_ds) != 3318 or len(class_names) != 25:
    raise RuntimeError("ROD Final A4 split/class count không khớp freeze")

E6_CONFIG = {"model": E6_MODEL, "init": "COCO_V2_DEFAULT", "epochs": 10,
             "batch": E6_BATCH, "seed": 0, "optimizer": "SGD", "lr": 0.005,
             "momentum": 0.9, "weight_decay": 0.0005, "scheduler": "StepLR(3,0.1)",
             "min_size": 640, "max_size": 960, "classes_with_background": 26,
             "train_count": len(train_ds), "valid_count": len(val_ds),
             "torch": torch.__version__, "torchvision": torchvision.__version__}
config_hash, dataset_hash = fingerprint(E6_CONFIG), sha256_file(source_yaml)
print("E6 config SHA256:", config_hash, "dataset YAML SHA256:", dataset_hash)
atomic_json(E6_WORK / "config.json", E6_CONFIG)
atomic_json(E6_WORK / "environment.json", {"torch": torch.__version__,
                                            "torchvision": torchvision.__version__,
                                            "gpu": torch.cuda.get_device_name(0)})
hf_token = UserSecretsClient().get_secret(HF_SECRET_NAME)
hf_api, repo_type = connect_hf(hf_token)
store = HFStore(hf_api, hf_token, repo_type, E6_FOLDER, E6_MODEL, config_hash,
                dataset_hash, E6_WORK / "hf_backup", max_epoch=30, compact=True)
restored = choose_resume(store, "E6_30epochs", "e4d667d8dff246d4be8b02fa5b2277682a46ee08cdce2c090d627467c115c2c5")
if restored is None:
    raise RuntimeError("No verified E6 recovery; refusing restart at epoch 1")

net = fasterrcnn_resnet50_fpn_v2(weights=None if restored else FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
                                 weights_backbone=None, min_size=640, max_size=960)
features = net.roi_heads.box_predictor.cls_score.in_features
net.roi_heads.box_predictor = FastRCNNPredictor(features, 26)
net.to("cuda")
optimizer = torch.optim.SGD(net.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
scaler = torch.amp.GradScaler("cuda")
current_epoch = completed_epoch = next_sample = 0
loss_sum, loss_count = 0.0, 0
metrics_rows, best_map = [], -1.0


def validate_e6_payload(saved, manifest):
    required = ("model", "optimizer", "scheduler", "scaler", "current_epoch", "completed_epoch",
                "next_sample", "loss_sum", "loss_count", "metrics_rows", "python_rng", "numpy_rng",
                "torch_rng", "cuda_rng", "config_hash", "dataset_hash")
    if any(key not in saved for key in required):
        raise RuntimeError("E6 checkpoint thiếu resume state")
    if saved["config_hash"] != config_hash or saved["dataset_hash"] != dataset_hash:
        raise RuntimeError("E6 checkpoint khác config/dataset")
    ep, done, offset = (saved[k] for k in ("current_epoch", "completed_epoch", "next_sample"))
    if (type(ep) is not int or type(done) is not int or type(offset) is not int or
            not 0 <= done <= E6_EPOCHS or not 0 <= ep <= E6_EPOCHS or
            not 0 <= offset <= len(train_ds) or ep != done or
            (offset and (offset % E6_BATCH != 0 and offset != len(train_ds))) or
            (done == E6_EPOCHS and (ep != E6_EPOCHS or offset != 0))):
        raise RuntimeError("E6 epoch/sample offset không hợp lệ")
    if manifest["completed_epoch"] != done or manifest.get("next_sample") != offset:
        raise RuntimeError("E6 checkpoint và HF manifest không nhất quán")
    if len(saved["metrics_rows"]) != done:
        raise RuntimeError("E6 metrics history thiếu epoch")
    if (not isinstance(saved["optimizer"], dict) or not saved["optimizer"].get("param_groups") or
            not isinstance(saved["scheduler"], dict) or saved["scheduler"].get("last_epoch") != done or
            not isinstance(saved["scaler"], dict)):
        raise RuntimeError("E6 optimizer/scheduler/AMP state không khớp số epoch")
    expected_batches = (offset + E6_BATCH - 1) // E6_BATCH
    if (type(saved["loss_count"]) is not int or saved["loss_count"] != expected_batches or
            not math.isfinite(float(saved["loss_sum"]))):
        raise RuntimeError("E6 mid-epoch loss/batch progress không nhất quán")
    return ep, done, offset


if restored:
    manifest, files = restored
    if "checkpoint.pt" not in files:
        raise RuntimeError("E6 HF snapshot thiếu checkpoint.pt")
    saved = torch.load(files["checkpoint.pt"], map_location="cpu", weights_only=False)
    current_epoch, completed_epoch, next_sample = validate_e6_payload(saved, manifest)
    net.load_state_dict(saved["model"])
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    scaler.load_state_dict(saved["scaler"])
    metrics_rows, best_map = saved["metrics_rows"], float(saved["best_map"])
    loss_sum, loss_count = float(saved["loss_sum"]), int(saved["loss_count"])
    random.setstate(saved["python_rng"])
    np.random.set_state(saved["numpy_rng"])
    torch.set_rng_state(saved["torch_rng"])
    torch.cuda.set_rng_state_all(saved["cuda_rng"])
    if "best_weights.pth" in files:
        import shutil
        shutil.copy2(files["best_weights.pth"], E6_WORK / "best_weights.pth")


def write_metrics():
    path = E6_WORK / "metrics.csv"
    with path.open("w", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=["epoch", "map50", "map5095", "mar100", "loss_mean"])
        writer.writeheader()
        writer.writerows(metrics_rows)
    return path


last_hf_sync = time.monotonic()
E6_HF_INTERVAL_SECONDS = 3600

def persist(ep, offset, done, running_loss, running_count):
    global last_hf_sync
    payload = {"model": net.state_dict(), "optimizer": optimizer.state_dict(),
               "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
               "current_epoch": int(ep), "next_sample": int(offset), "completed_epoch": int(done),
               "loss_sum": float(running_loss), "loss_count": int(running_count),
               "best_map": float(best_map), "metrics_rows": metrics_rows,
               "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
               "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(),
               "config_hash": config_hash, "dataset_hash": dataset_hash}
    path, temp = E6_WORK / "checkpoint.pt", E6_WORK / "checkpoint.pt.tmp"
    torch.save(payload, temp)
    os.replace(temp, path)
    if (offset > 0 and time.monotonic() < E6_DEADLINE
        and time.monotonic() - last_hf_sync < E6_HF_INTERVAL_SECONDS):
        print("E6 local recovery saved; HF waits for epoch end/hour/deadline", flush=True)
        return
    files = {"checkpoint.pt": path, "metrics.csv": write_metrics(),
             "config.json": E6_WORK / "config.json",
             "environment.json": E6_WORK / "environment.json"}
    if (E6_WORK / "best_weights.pth").is_file():
        files["best_weights.pth"] = E6_WORK / "best_weights.pth"
    latest = store.save(files, done, extra={"next_sample": int(offset), "current_epoch": int(ep),
                                           "class_count": 25, "architecture": E6_MODEL})
    atomic_json(E6_WORK / "last_hf_manifest.json", latest)
    if offset == 0 and done in (10,20,30):
        import shutil
        shutil.copy2(path, E6_WORK / ("resume_epoch%03d.pt" % done))
    last_hf_sync = time.monotonic()


@torch.inference_mode()
def evaluate():
    evaluator = MeanAveragePrecision(iou_type="bbox")
    loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, collate_fn=collate,
                        generator=torch.Generator().manual_seed(8000))
    net.eval()
    valid_seen = 0
    valid_started = valid_last_log = time.monotonic()
    print(f"E6 VALID start: {len(val_ds)} images", flush=True)
    for images, targets in loader:
        predictions = net([image.to("cuda") for image in images])
        evaluator.update([{k: v.detach().cpu() for k, v in pred.items() if k in ("boxes", "labels", "scores")}
                          for pred in predictions],
                         [{k: v.detach().cpu() for k, v in target.items() if k in ("boxes", "labels")}
                          for target in targets])
        valid_seen += len(images)
        now = time.monotonic()
        if valid_seen == 1 or valid_seen == len(val_ds) or now - valid_last_log >= 60:
            print(f"E6 VALID {valid_seen}/{len(val_ds)} images "
                  f"({100 * valid_seen / len(val_ds):.1f}%) elapsed={now - valid_started:.0f}s", flush=True)
            valid_last_log = now
    print("E6 VALID computing mAP...", flush=True)
    values = evaluator.compute()
    return float(values["map_50"]), float(values["map"]), float(values["mar_100"])


if completed_epoch == E6_EPOCHS:
    print("E6 COMPLETED 30/30 theo HF manifest và checkpoint; không train")
else:
    if not restored:
        persist(0, 0, 0, 0.0, 0)
    print("E6 resume state: completed_epoch=", completed_epoch,
          "current_epoch=", current_epoch, "next_sample=", next_sample,
          "HF backup interval (batches)=", E6_BACKUP_BATCHES)
    stop = False
    for ep in range(current_epoch, E6_EPOCHS):
        permutation = np.random.default_rng(10000 + ep).permutation(len(train_ds)).tolist()
        offset = next_sample if ep == current_epoch else 0
        running_loss = loss_sum if ep == current_epoch else 0.0
        running_count = loss_count if ep == current_epoch else 0
        subset = Subset(train_ds, permutation[offset:])
        loader = DataLoader(subset, batch_size=E6_BATCH, shuffle=False, num_workers=2, collate_fn=collate,
                            pin_memory=True, generator=torch.Generator().manual_seed(20000 + ep))
        net.train()
        epoch_started = train_last_log = time.monotonic()
        session_batches = 0
        start_offset = offset
        print(f"E6 TRAIN epoch {ep + 1}/{E6_EPOCHS} start: "
              f"samples={offset}/{len(train_ds)} lr={optimizer.param_groups[0]['lr']:.6g}", flush=True)
        for images, targets in loader:
            images = [image.to("cuda", non_blocking=True) for image in images]
            targets = [{key: val.to("cuda", non_blocking=True) for key, val in target.items()} for target in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                parts = net(images, targets)
                loss = sum(parts.values())
            if not torch.isfinite(loss):
                raise RuntimeError("E6 loss NaN/Inf")
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.detach().cpu())
            running_count += 1
            offset += len(images)
            session_batches += 1
            now = time.monotonic()
            if session_batches == 1 or offset == len(train_ds) or now - train_last_log >= 60:
                elapsed = max(now - epoch_started, 1e-6)
                rate = (offset - start_offset) / elapsed
                eta = (len(train_ds) - offset) / max(rate, 1e-6)
                print(f"E6 TRAIN epoch {ep + 1}/{E6_EPOCHS} "
                      f"samples={offset}/{len(train_ds)} ({100 * offset / len(train_ds):.1f}%) "
                      f"batch={running_count} loss_mean={running_loss / running_count:.4f} "
                      f"elapsed={elapsed:.0f}s ETA={eta:.0f}s", flush=True)
                train_last_log = now
            if running_count % E6_BACKUP_BATCHES == 0 or time.monotonic() >= E6_DEADLINE:
                persist(ep, offset, ep, running_loss, running_count)
            if time.monotonic() >= E6_DEADLINE:
                print("E6 dừng sau batch đã lưu HF; tiếp tục bằng Edit > Save Version > Save & Run All")
                stop = True
                break
        if stop:
            break
        if offset != len(train_ds):
            raise RuntimeError("E6 epoch chưa xử lý hết TRAIN")
        score50, score, recall = evaluate()
        scheduler.step()
        if score > best_map:
            best_map = score
            torch.save(net.state_dict(), E6_WORK / "best_weights.pth")
        metrics_rows.append({"epoch": ep + 1, "map50": score50, "map5095": score,
                             "mar100": recall, "loss_mean": running_loss / running_count})
        print(f"E6 epoch {ep + 1}/{E6_EPOCHS} metrics: "
              f"mAP50={score50:.6f} mAP50-95={score:.6f} mAR100={recall:.6f} "
              f"best_mAP50-95={best_map:.6f}; saving checkpoint...", flush=True)
        completed_epoch = ep + 1
        persist(completed_epoch, 0, completed_epoch, 0.0, 0)
        print("E6 completed epoch", completed_epoch, "/30; VALID mAP50-95", score, flush=True)
        if time.monotonic() >= E6_DEADLINE:
            break

latest = store.restore()
if latest is None:
    raise RuntimeError("E6 chưa có HF checkpoint sau phiên này")
latest_manifest, latest_files = latest
latest_payload = torch.load(latest_files["checkpoint.pt"], map_location="cpu", weights_only=False)
validate_e6_payload(latest_payload, latest_manifest)
print("E6 status:", latest_manifest["completed_epoch"], "/30",
      "COMPLETE" if latest_manifest["is_complete"] else "CONTINUE",
      "next_sample", latest_manifest["next_sample"])


## Kết quả
Kiểm tra completed_epoch=30, is_complete=true và checksum. Save Version/Download Output.
Giữ mốc 10 cũ và resume_epoch020.pt/resume_epoch030.pt trong Output; mỗi snapshot HF bất biến cũng giữ tiến độ tương ứng.
Không dùng TEST trong training. Không reset LR; E6 giữ StepLR cũ nên LR tiếp tục rất nhỏ.
